# Pothole Detection — Phase 0 & 1: Colab Setup + Dataset Acquisition

This notebook is part of the PFPD road-hazard-detection project. This first notebook covers:
- **Phase 0**: basic Colab setup (GPU, Google Drive)
- **Phase 1**: installing tools + downloading pothole datasets (Kaggle + Roboflow)

**Before running anything:**
1. Go to `Runtime` → `Change runtime type` → set Hardware accelerator to **GPU** (T4 is free; with Colab Pro you can pick A100/L4 under `Runtime` → `Change runtime type` → `Premium GPU`).
2. Run cells **top to bottom**, one at a time (Shift+Enter). Don't skip ahead.


## Step 1 — Confirm you have a GPU
If this errors out or shows no GPU, go back to `Runtime > Change runtime type` and select GPU, then re-run.

In [ ]:
!nvidia-smi

## Step 2 — Mount Google Drive

Since you have Colab Pro (5TB), we'll store everything in Drive so it survives even after the Colab session disconnects (Colab's local disk is wiped every session — Drive is not).

Running this cell will pop up a Google sign-in/authorization window — click through it and allow access.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = "/content/drive/MyDrive/pothole_detection"
os.makedirs(PROJECT_DIR, exist_ok=True)
print("Project folder ready at:", PROJECT_DIR)

## Step 3 — Install the libraries we need
- `ultralytics` → YOLO training/eval
- `kagglehub` → pull datasets from Kaggle
- `roboflow` → pull datasets from Roboflow Universe

In [ ]:
!pip install -q ultralytics kagglehub roboflow

## Step 4 — Dataset source #1: Kaggle ("Annotated Potholes Image Dataset", 665 images)

This is a well-known, clean pothole dataset (by Atikur Rahman Chitholian) — good as our base.

**One-time setup (only needed the first time you run this notebook):**
1. Go to https://www.kaggle.com → click your profile picture → **Settings**.
2. Scroll to **API** → click **Create New Token**. This downloads a file called `kaggle.json`.
3. Run the cell below — it will ask you to upload that file. Select `kaggle.json`.

You only need to do this once per Colab session (you'll need to re-upload if your session disconnects).

In [ ]:
from google.colab import files
print("Upload your kaggle.json (from kaggle.com -> Settings -> API -> Create New Token)")
uploaded = files.upload()

import os
os.makedirs("/root/.kaggle", exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json
print("Kaggle credentials saved.")

In [ ]:
import kagglehub

kaggle_path = kagglehub.dataset_download("andrewmvd/pothole-detection")
print("Kaggle dataset downloaded to:", kaggle_path)

## Step 5 — Dataset source #2: Roboflow Universe

Roboflow Universe hosts thousands of community pothole datasets, already labeled in YOLO format — much easier to merge than raw images.

**One-time setup:**
1. Go to https://app.roboflow.com → sign up (free) → **Settings** → **API Keys** (or directly https://app.roboflow.com/settings/api).
2. Copy your **Private API Key** and paste it below where it says `PASTE_YOUR_KEY_HERE`.

`ROBOFLOW_DATASETS` below is a list you can keep adding to — every time you (or I) find a good pothole dataset on Roboflow Universe, add its workspace/project/version here and re-run the cell. This is how we do the "search and merge multiple datasets" part, deliberately rather than blindly.

In [ ]:
from roboflow import Roboflow

ROBOFLOW_API_KEY = "PASTE_YOUR_KEY_HERE"  # https://app.roboflow.com/settings/api

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

# Add more datasets here as (workspace, project, version) - check the dataset's Roboflow Universe
# page for the exact slugs and available version numbers before adding.
ROBOFLOW_DATASETS = [
    {"workspace": "projects-hjaax", "project": "pothole-detection-clybc", "version": 1},
]

roboflow_dataset_paths = []
for ds in ROBOFLOW_DATASETS:
    try:
        project = rf.workspace(ds["workspace"]).project(ds["project"])
        dataset = project.version(ds["version"]).download("yolov8")
        roboflow_dataset_paths.append(dataset.location)
        print("Downloaded:", dataset.location)
    except Exception as e:
        print(f"Skipped {ds} - couldn't download ({e}). Check the version number on its Universe page.")

## Step 6 — Sanity check: see what we actually downloaded
This just prints the folder structure so we can confirm both datasets landed correctly before we merge them in the next notebook (Phase 2).

In [ ]:
import os

def show_tree(path, prefix="", max_depth=2, depth=0):
    if depth > max_depth or not os.path.isdir(path):
        return
    for item in sorted(os.listdir(path))[:15]:
        print(prefix + item)
        full = os.path.join(path, item)
        if os.path.isdir(full):
            show_tree(full, prefix + "  ", max_depth, depth + 1)

print("== Kaggle dataset ==")
show_tree(kaggle_path)

print("\n== Roboflow dataset(s) ==")
for p in roboflow_dataset_paths:
    print(f"-- {p} --")
    show_tree(p)

## Step 7 — Copy raw downloads into your Drive project folder

So they persist and Phase 2 (the merge notebook) can find them without re-downloading.

In [ ]:
import shutil, os

raw_dir = os.path.join(PROJECT_DIR, "raw_datasets")
os.makedirs(raw_dir, exist_ok=True)

def copy_dataset(src, name):
    dst = os.path.join(raw_dir, name)
    if os.path.exists(dst):
        print(f"{name} already copied, skipping.")
        return
    shutil.copytree(src, dst)
    print(f"Copied {name} -> {dst}")

copy_dataset(kaggle_path, "kaggle_andrewmvd_pothole")
for i, p in enumerate(roboflow_dataset_paths):
    copy_dataset(p, f"roboflow_dataset_{i+1}")

print("\nAll raw datasets are now saved in:", raw_dir)

---
### What's next (Phase 2)

In the next notebook we'll:
- Convert both datasets into one unified YOLO folder structure (`images/`, `labels/`, single `data.yaml`)
- Fix any class-name mismatches (e.g. "pothole" vs "Pothole" vs "0")
- Split into train/val/test
- Visualize a few samples to confirm labels are correct before training

Let me know once Steps 1–7 above have run successfully (or if any step errored) and we'll build that next.